In [1]:
import pandas as pd
import numpy as np

from pathlib import Path

In [2]:
DATA_PATH = Path("../data/raw/PS_20174392719_1491204439457_log.csv")

df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
df.head()

Shape: (6362620, 11)


,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0


In [3]:
base_features = [
    "step",
    "type",
    "amount",
    "oldbalanceOrg",
    "newbalanceOrig",
    "oldbalanceDest",
    "newbalanceDest",
    "isFlaggedFraud",
    "isFraud"
]

feature_df = df[base_features].copy()

#### Origin balance change

##### The sender's balance changes from:

In [4]:
feature_df["balance_change_orig"] =(
    feature_df["oldbalanceOrg"] - feature_df["newbalanceOrig"]
) 

In [5]:
feature_df["balance_change_dest"] = (
    feature_df["newbalanceDest"]
    - feature_df["oldbalanceDest"]
)

##### Origin balance error

##### A useful fraud signal is whether:
old sender balance
       -
new sender balance

In [6]:
feature_df["balance_error_orig"] = (
    feature_df["oldbalanceOrg"]
    - feature_df["amount"]
    - feature_df["newbalanceOrig"]
)

##### Destination balance error

In [7]:
feature_df["balance_error_dest"] = (
    feature_df["oldbalanceDest"]
    + feature_df["amount"]
    - feature_df["newbalanceDest"]
)

##### Amount-to-origin-balance ratio

In [8]:
feature_df["amount_to_old_balance_ratio"] = (
    feature_df["amount"]
    / (feature_df["oldbalanceOrg"] + 1)
)

##### Amount-to-new-balance ratio

In [9]:
feature_df["amount_to_new_balance_ratio"] = (
    feature_df["amount"]
    / (feature_df["newbalanceOrig"] + 1)
)

##### Destination amount ratio

In [10]:
feature_df["amount_to_old_dest_balance_ratio"] = (
    feature_df["amount"]
    / (feature_df["oldbalanceDest"] + 1)
)

##### Time features

PaySim's step represents simulation time.

a rough hour-of-day representation:

In [11]:
feature_df["transaction_hour"] = (
    feature_df["step"] % 24
)

feature_df["transaction_day"] = (
    feature_df["step"] // 24
)

In [12]:
new_features = [
    "balance_change_orig",
    "balance_change_dest",
    "balance_error_orig",
    "balance_error_dest",
    "amount_to_old_balance_ratio",
    "amount_to_new_balance_ratio",
    "amount_to_old_dest_balance_ratio",
    "transaction_hour",
    "transaction_day"
]

feature_df[new_features].describe().T

,count,mean,std,min,25%,50%,75%,max
balance_change_orig,6362620.0,-21230.564504,146643.286487,-1915267.90,0.000000,0.000000,10150.435,1.000000e+07
balance_change_dest,6362620.0,124294.731682,812939.081073,-13060826.21,0.000000,0.000000,149105.425,1.056878e+08
balance_error_orig,6362620.0,-201092.468053,606650.460140,-92445516.64,-249641.087500,-68677.255000,-2954.230,1.000000e-02
balance_error_dest,6362620.0,55567.171867,441528.768827,-75885725.63,0.000000,3500.490000,29353.045,1.319123e+07
amount_to_old_balance_ratio,6362620.0,70674.476792,508424.269379,0.00,0.234401,6.453832,12287.765,9.244552e+07
amount_to_new_balance_ratio,6362620.0,136613.064270,604439.687607,0.00,0.261557,7557.495000,134263.970,9.244552e+07
amount_to_old_dest_balance_ratio,6362620.0,27883.832244,187930.639208,0.00,0.160704,0.917831,9732.300,6.096528e+07
transaction_hour,6362620.0,15.321454,4.321799,0.00,12.000000,16.000000,19.000,2.300000e+01
transaction_day,6362620.0,9.503158,5.922111,0.00,6.000000,9.000000,13.000,3.000000e+01


##### Check for infinite values

Ratios can sometimes produce infinite values if the denominator is zero.

In [13]:
np.isinf(
    feature_df[new_features]
).sum()

balance_change_orig                 0
balance_change_dest                 0
balance_error_orig                  0
balance_error_dest                  0
amount_to_old_balance_ratio         0
amount_to_new_balance_ratio         0
amount_to_old_dest_balance_ratio    0
transaction_hour                    0
transaction_day                     0
dtype: int64

In [14]:
feature_df.replace(
    [np.inf, -np.inf],
    np.nan,
    inplace=True
)

In [15]:
print(
    feature_df[new_features]
    .isnull()
    .sum()
)

balance_change_orig                 0
balance_change_dest                 0
balance_error_orig                  0
balance_error_dest                  0
amount_to_old_balance_ratio         0
amount_to_new_balance_ratio         0
amount_to_old_dest_balance_ratio    0
transaction_hour                    0
transaction_day                     0
dtype: int64


In [16]:
feature_comparison = (
    feature_df
    .groupby("isFraud")[new_features]
    .median()
)

feature_comparison.T

isFraud,0,1
balance_change_orig,0.000000,436317.490000
balance_change_dest,0.000000,0.000000
balance_error_orig,-69049.310000,0.000000
balance_error_dest,3500.680000,2231.460000
amount_to_old_balance_ratio,6.511566,0.999998
amount_to_new_balance_ratio,7487.060000,401529.350000
amount_to_old_dest_balance_ratio,0.915134,116419.580000
transaction_hour,16.000000,12.000000
transaction_day,9.000000,15.000000


In [17]:
correlation_features = [
    "amount",
    "oldbalanceOrg",
    "newbalanceOrig",
    "oldbalanceDest",
    "newbalanceDest",
    *new_features
]

correlation = feature_df[
    correlation_features + ["isFraud"]
].corr()["isFraud"].sort_values(
    ascending=False
)

print(correlation)

isFraud                             1.000000
balance_change_orig                 0.362472
amount_to_old_dest_balance_ratio    0.180205
amount                              0.076688
amount_to_new_balance_ratio         0.068333
balance_error_dest                  0.055120
transaction_day                     0.032577
balance_change_dest                 0.027028
balance_error_orig                  0.011283
oldbalanceOrg                       0.010154
newbalanceDest                      0.000535
amount_to_old_balance_ratio        -0.004915
oldbalanceDest                     -0.005885
newbalanceOrig                     -0.008148
transaction_hour                   -0.031403
Name: isFraud, dtype: float64


In [18]:
print("Original columns:", len(base_features))
print("Engineered columns:", len(new_features))
print("Total columns:", feature_df.shape[1])

feature_df.head()

Original columns: 9
Engineered columns: 9
Total columns: 18


,step,type,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFlaggedFraud,isFraud,balance_change_orig,balance_change_dest,balance_error_orig,balance_error_dest,amount_to_old_balance_ratio,amount_to_new_balance_ratio,amount_to_old_dest_balance_ratio,transaction_hour,transaction_day
0,1,PAYMENT,9839.64,170136.0,160296.36,0.0,0.0,0,0,9839.64,0.0,0.0,9839.64,0.057834,0.061384,9839.640000,1,0
1,1,PAYMENT,1864.28,21249.0,19384.72,0.0,0.0,0,0,1864.28,0.0,0.0,1864.28,0.087731,0.096168,1864.280000,1,0
2,1,TRANSFER,181.00,181.0,0.00,0.0,0.0,0,1,181.00,0.0,0.0,181.00,0.994505,181.000000,181.000000,1,0
3,1,CASH_OUT,181.00,181.0,0.00,21182.0,0.0,0,1,181.00,-21182.0,0.0,21363.00,0.994505,181.000000,0.008545,1,0
4,1,PAYMENT,11668.14,41554.0,29885.86,0.0,0.0,0,0,11668.14,0.0,0.0,11668.14,0.280788,0.390410,11668.140000,1,0


In [19]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
import joblib


# ============================================================
# 1. Separate features and target
# ============================================================

TARGET = "isFraud"

X = feature_df.drop(columns=[TARGET])
y = feature_df[TARGET]

print("X shape:", X.shape)
print("y shape:", y.shape)


# ============================================================
# 2. Train / Test Split
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("\nTrain/Test Split")
print("----------------")
print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)


# ============================================================
# 3. Define feature types
# ============================================================

numeric_features = [
    "step",
    "amount",
    "oldbalanceOrg",
    "newbalanceOrig",
    "oldbalanceDest",
    "newbalanceDest",
    "isFlaggedFraud",
    "balance_change_orig",
    "balance_change_dest",
    "balance_error_orig",
    "balance_error_dest",
    "amount_to_old_balance_ratio",
    "amount_to_new_balance_ratio",
    "amount_to_old_dest_balance_ratio",
    "transaction_hour",
    "transaction_day"
]

categorical_features = [
    "type"
]


# ============================================================
# 4. Numeric preprocessing
# ============================================================

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])


# ============================================================
# 5. Categorical preprocessing
# ============================================================

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])


# ============================================================
# 6. Combine preprocessing
# ============================================================

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])


# ============================================================
# 7. Fit only on training data
# ============================================================

X_train_processed = preprocessor.fit_transform(X_train)

X_test_processed = preprocessor.transform(X_test)


print("\nProcessed Data")
print("--------------")
print("X_train_processed:", X_train_processed.shape)
print("X_test_processed :", X_test_processed.shape)


# ============================================================
# 8. Create directories
# ============================================================

PROCESSED_DIR = Path("../data/processed")
PREPROCESSOR_DIR = Path("../artifacts/preprocessors")

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
PREPROCESSOR_DIR.mkdir(parents=True, exist_ok=True)


# ============================================================
# 9. Save processed datasets
# ============================================================

np.save(
    PROCESSED_DIR / "X_train_engineered.npy",
    X_train_processed
)

np.save(
    PROCESSED_DIR / "X_test_engineered.npy",
    X_test_processed
)

np.save(
    PROCESSED_DIR / "y_train_engineered.npy",
    y_train.to_numpy()
)

np.save(
    PROCESSED_DIR / "y_test_engineered.npy",
    y_test.to_numpy()
)


# ============================================================
# 10. Save preprocessor
# ============================================================

joblib.dump(
    preprocessor,
    PREPROCESSOR_DIR / "engineered_preprocessor.pkl"
)


# ============================================================
# 11. Final verification
# ============================================================

print("\nSaved successfully.")
print("------------------")

print(
    "X_train_engineered.npy:",
    PROCESSED_DIR / "X_train_engineered.npy"
)

print(
    "X_test_engineered.npy:",
    PROCESSED_DIR / "X_test_engineered.npy"
)

print(
    "y_train_engineered.npy:",
    PROCESSED_DIR / "y_train_engineered.npy"
)

print(
    "y_test_engineered.npy:",
    PROCESSED_DIR / "y_test_engineered.npy"
)

print(
    "engineered_preprocessor.pkl:",
    PREPROCESSOR_DIR / "engineered_preprocessor.pkl"
)

X shape: (6362620, 17)
y shape: (6362620,)

Train/Test Split
----------------
X_train: (5090096, 17)
X_test : (1272524, 17)
y_train: (5090096,)
y_test : (1272524,)

Processed Data
--------------
X_train_processed: (5090096, 21)
X_test_processed : (1272524, 21)

Saved successfully.
------------------
X_train_engineered.npy: ..\data\processed\X_train_engineered.npy
X_test_engineered.npy: ..\data\processed\X_test_engineered.npy
y_train_engineered.npy: ..\data\processed\y_train_engineered.npy
y_test_engineered.npy: ..\data\processed\y_test_engineered.npy
engineered_preprocessor.pkl: ..\artifacts\preprocessors\engineered_preprocessor.pkl
